# Federal vs. state fiscal impact of Medicaid reforms

PolicyEngine-US attributes Medicaid and CHIP spending between the federal government and states using the statutory FMAP and enhanced FMAP formulas (42 U.S.C. § 1396d(b), § 1396d(y), § 1397ee(b)). This notebook demonstrates how to use the resulting `federal_benefit_cost` and `state_benefit_cost` variables to produce federal vs. state fiscal impact estimates for Medicaid reforms.

Primary references:
- Microsim variables: https://github.com/PolicyEngine/policyengine-us/pull/8076
- FY2026 FMAP values: 89 FR 94742 (Nov 29 2024)
- Expansion FMAP: ACA § 2001(a)(3) — 90% for newly eligible adults 2020+

In [ ]:
from policyengine_us import Microsimulation
from policyengine_core.reforms import Reform
import pandas as pd
import matplotlib.pyplot as plt

YEAR = 2026

## Baseline: current-law Medicaid

Compute national totals for `federal_benefit_cost` and `state_benefit_cost` under current law, which sum Medicaid and CHIP federal/state shares per enrollee.

In [ ]:
baseline = Microsimulation()

fed_baseline = baseline.calculate('federal_benefit_cost', YEAR).sum()
state_baseline = baseline.calculate('state_benefit_cost', YEAR).sum()

print(f'Baseline federal Medicaid+CHIP cost: ${fed_baseline / 1e9:,.1f}B')
print(f'Baseline state Medicaid+CHIP cost:   ${state_baseline / 1e9:,.1f}B')
print(f'Total:                               ${(fed_baseline + state_baseline) / 1e9:,.1f}B')

## Reform: repeal ACA expansion FMAP (90% → state regular FMAP)

Under current law, expansion adults are matched at 90% federal. Setting the expansion FMAP to 0 would cause the variable's `defined_for` gate to zero out expansion adult cost shares entirely — instead, we model the more realistic policy of reverting expansion adults to the regular state FMAP (i.e., treating them like traditional Medicaid enrollees for cost-share purposes).

In [ ]:
# Repeal the 90% expansion FMAP for 2026+: set expansion_fmap to 0, which
# falls through in medicaid_federal_share to the regular FMAP path for
# expansion adults (matching traditional Medicaid cost sharing).
expansion_repeal = Reform.from_dict({
    'gov.hhs.medicaid.cost_share.expansion_fmap': {
        f'{YEAR}-01-01.{YEAR}-12-31': 0.0,
    }
})

reformed = Microsimulation(reform=expansion_repeal)
fed_reform = reformed.calculate('federal_benefit_cost', YEAR).sum()
state_reform = reformed.calculate('state_benefit_cost', YEAR).sum()

fed_change = fed_reform - fed_baseline
state_change = state_reform - state_baseline

print(f'Federal cost change: ${fed_change / 1e9:+,.1f}B')
print(f'State cost change:   ${state_change / 1e9:+,.1f}B')
print(f'Total cost change:   ${(fed_change + state_change) / 1e9:+,.1f}B')

## Per-state breakdown

Which states bear the most of the shifted cost? Expansion states with low regular FMAP (high per-capita income) see the biggest increase in state share — e.g., California goes from 10% to 50%, Mississippi from 10% to ~23% (1 − 0.7690).

In [ ]:
by_state = pd.DataFrame({
    'state': baseline.calculate('state_code', YEAR).values,
    'fed_change': (reformed.calculate('federal_benefit_cost', YEAR).values
                   - baseline.calculate('federal_benefit_cost', YEAR).values),
    'state_change': (reformed.calculate('state_benefit_cost', YEAR).values
                     - baseline.calculate('state_benefit_cost', YEAR).values),
    'weight': baseline.calculate('household_weight', YEAR, map_to='person').values,
})
by_state['fed_change_weighted'] = by_state['fed_change'] * by_state['weight']
by_state['state_change_weighted'] = by_state['state_change'] * by_state['weight']
state_agg = (by_state.groupby('state')[['fed_change_weighted', 'state_change_weighted']]
             .sum() / 1e9)
state_agg.columns = ['Federal ($B)', 'State ($B)']
state_agg['Total ($B)'] = state_agg.sum(axis=1)
state_agg.sort_values('State ($B)', ascending=False).head(10)

## Interpretation

The aggregate cost of expansion Medicaid (~$100B/year) stays the same — the population enrolled doesn't change. But the **attribution** shifts dramatically: the federal government saves roughly 80% of expansion cost, and states pick up the other ~80%.

Reporting only "total Medicaid cost change" would show ~$0 for this reform, which misses the entire policy story. The federal/state split is what drives the actual political economy of Medicaid reform — which is why scoring reforms by level of government is the correct lens.

Same pattern applies to:
- SNAP benefit state match under OBBBA § 10105 (FY2028+, error-rate-tiered)
- CHIP program changes (uses enhanced FMAP per 42 U.S.C. § 1397ee(b))
- Any FMAP-linked program (foster care, adoption assistance, CCDF match)